# CatBoost：原生类别变量的 Boosting 挑战者

本 Notebook 使用与共同输入 Logistic、随机森林完全相同的 126 个公共特征和固定 Train/Selection 切分。CatBoost 原生处理物理类别变量与数值缺失，不做 One-Hot。候选方案是预先声明的 6 组代表性配置；若接近最优的候选触及 3,000 轮边界，才允许延长至 5,000 轮。Selection 仅用于 early stopping、锁模和与共同输入 Logistic 的配对比较，Test 保持封存。锁模后的全局 SHAP 只在 Selection 分层抽样上计算。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'configs' / 'experiment.yaml').is_file()
)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from credit_risk.comparison import load_locked_selection_prediction
from credit_risk.development import load_development_data
from credit_risk.models.catboost import run_catboost
print('Project root:', PROJECT_ROOT)

Project root: /Users/ganlu/Desktop/home-credit-risk-english


In [2]:
development = load_development_data(PROJECT_ROOT)
assert not hasattr(development, 'X_test')
assert not hasattr(development, 'y_test')
reference_selection_prediction = load_locked_selection_prediction(
    development, 'common_logistic'
)
display(development.split_overview)
print('公共特征数:', len(development.feature_columns))
print('已对齐的共同输入 Logistic Selection 预测数:', len(reference_selection_prediction))

,sample,n,bad_rate
0,Train,215257,0.080727
1,Selection,46127,0.080734
2,Test (sealed),46127,NaN


公共特征数: 126
已对齐的共同输入 Logistic Selection 预测数: 46127


In [3]:
search_settings = development.configs['model_search_spaces']['catboost']
display(pd.DataFrame(search_settings['candidate_profiles']))
print('基础迭代上限:', search_settings['max_iterations'])
print('边界延长上限:', search_settings['extension_max_iterations'])
print('Early stopping patience:', search_settings['early_stopping_rounds'])

,name,depth,learning_rate,l2_leaf_reg,subsample,rsm,random_strength
0,legacy_conservative,4,0.02,5.0,0.7,0.8,0.5
1,shallow_faster,4,0.03,5.0,0.8,0.9,0.5
2,balanced,5,0.03,5.0,0.8,0.9,1.0
3,legacy_baseline,6,0.03,5.0,0.8,0.9,1.0
4,faster_strong_l2,6,0.05,10.0,0.8,0.9,1.0
5,deep_regularized,7,0.03,10.0,0.7,0.8,2.0


基础迭代上限: 3000
边界延长上限: 5000
Early stopping patience: 150


In [4]:
result = run_catboost(
    development,
    reference_selection_prediction=reference_selection_prediction,
    register_lock=True,
)
print('CatBoost 搜索、固定迭代重训与锁模完成。')

CatBoost candidate 1 [base_search/legacy_conservative] Selection AR=0.530366, gap=0.034045, trees=3000/3000, fit=173.6s


CatBoost candidate 2 [base_search/shallow_faster] Selection AR=0.533774, gap=0.051339, trees=3000/3000, fit=216.3s


CatBoost candidate 3 [base_search/balanced] Selection AR=0.535077, gap=0.079779, trees=2996/3000, fit=273.8s


CatBoost candidate 4 [base_search/legacy_baseline] Selection AR=0.535672, gap=0.120602, trees=2936/3000, fit=325.6s


CatBoost candidate 5 [base_search/faster_strong_l2] Selection AR=0.532774, gap=0.084305, trees=1271/1421, fit=150.2s


CatBoost candidate 6 [base_search/deep_regularized] Selection AR=0.534447, gap=0.123268, trees=2190/2340, fit=247.6s


CatBoost candidate 7 [boundary_extension/shallow_faster] Selection AR=0.534763, gap=0.063361, trees=3770/3920, fit=275.9s


CatBoost candidate 8 [boundary_extension/legacy_baseline] Selection AR=0.535772, gap=0.129458, trees=3187/3337, fit=369.5s


CatBoost candidate 9 [boundary_extension/balanced] Selection AR=0.535116, gap=0.081490, trees=3063/3213, fit=309.2s


CatBoost 搜索、固定迭代重训与锁模完成。


In [5]:
display(result.search_results)
display(result.metrics)
display(result.uplift_vs_logistic)
display(result.feature_importance.head(30))
display(result.shap_importance.head(30))
display(result.selection_deciles)
display(result.locked_config)
print('锁模产物:', result.run_directory)

,candidate,stage,profile,max_iterations,attempted_iterations,best_iteration,reached_iteration_boundary,depth,learning_rate,l2_leaf_reg,...,selection_ar,ar_gap,train_ks,selection_ks,selection_ap,selection_logloss,capacity,fit_seconds,parameters_json,eligible_for_lock
0,8,boundary_extension,legacy_baseline,5000,3337,3187,False,6,0.03,5.0,...,0.535772,0.129458,0.503720,0.400020,0.255385,0.243243,29371.392,369.513223,"{""allow_writing_files"": false, ""bootstrap_type...",True
1,4,base_search,legacy_baseline,3000,3000,2936,True,6,0.03,5.0,...,0.535672,0.120602,0.496694,0.400659,0.254902,0.243287,27058.176,325.557705,"{""allow_writing_files"": false, ""bootstrap_type...",False
2,9,boundary_extension,balanced,5000,3213,3063,False,5,0.03,5.0,...,0.535116,0.081490,0.463172,0.399782,0.256170,0.243344,14114.304,309.162215,"{""allow_writing_files"": false, ""bootstrap_type...",True
3,3,base_search,balanced,3000,3000,2996,True,5,0.03,5.0,...,0.535077,0.079779,0.461671,0.399579,0.255992,0.243360,13805.568,273.792154,"{""allow_writing_files"": false, ""bootstrap_type...",False
4,7,boundary_extension,shallow_faster,5000,3920,3770,False,4,0.03,5.0,...,0.534763,0.063361,0.450639,0.399164,0.254813,0.243419,8686.080,275.873355,"{""allow_writing_files"": false, ""bootstrap_type...",True
5,6,base_search,deep_regularized,3000,2340,2190,False,7,0.03,10.0,...,0.534447,0.123268,0.497020,0.395894,0.253146,0.243544,15697.920,247.559197,"{""allow_writing_files"": false, ""bootstrap_type...",True
6,2,base_search,shallow_faster,3000,3000,3000,True,4,0.03,5.0,...,0.533774,0.051339,0.440325,0.398238,0.255071,0.243529,6912.000,216.297662,"{""allow_writing_files"": false, ""bootstrap_type...",False
7,5,base_search,faster_strong_l2,3000,1421,1271,False,6,0.05,10.0,...,0.532774,0.084305,0.463713,0.396771,0.253234,0.243729,5856.768,150.193435,"{""allow_writing_files"": false, ""bootstrap_type...",True
8,1,base_search,legacy_conservative,3000,3000,3000,True,4,0.02,5.0,...,0.530366,0.034045,0.422768,0.392718,0.253593,0.244042,5376.000,173.579842,"{""allow_writing_files"": false, ""bootstrap_type...",True


,sample,n,bad_rate,AUC,AR,KS,AP,Logloss
0,Train,215257,0.080727,0.799062,0.598124,0.450639,0.320145,0.232412
1,Selection,46127,0.080734,0.767382,0.534763,0.399164,0.254813,0.243419


,model,reference_model,metric,difference,lower,upper,bootstrap_se,confidence_level,bootstrap_replicates,higher_is_better
0,catboost,common_logistic,AUC,0.019239,0.015842,0.022299,0.001735,0.95,500,True
1,catboost,common_logistic,AR,0.038478,0.031684,0.044599,0.003470,0.95,500,True
2,catboost,common_logistic,KS,0.032687,0.021785,0.043158,0.005411,0.95,500,True
3,catboost,common_logistic,AP,0.019383,0.013321,0.026195,0.003330,0.95,500,True
4,catboost,common_logistic,Logloss,-0.005257,-0.006220,-0.004281,0.000482,0.95,500,False


,feature,importance
0,EXT_SOURCE_3,0.182875
1,EXT_SOURCE_2,0.151178
2,CREDIT_ANNUITY_RATIO,0.101194
3,EXT_SOURCE_1,0.067522
4,AGE_YEARS,0.051737
5,GOODS_CREDIT_RATIO,0.038697
6,AMT_GOODS_PRICE,0.025286
7,AMT_ANNUITY,0.024136
8,NAME_EDUCATION_TYPE,0.019434
9,ID_PUBLISH_YEARS,0.017086


,feature,mean_abs_shap
0,EXT_SOURCE_3,0.349975
1,EXT_SOURCE_2,0.298812
2,EXT_SOURCE_1,0.190145
3,CREDIT_ANNUITY_RATIO,0.146951
4,EMPLOYED_YEARS,0.144488
5,AMT_ANNUITY,0.120440
6,GOODS_CREDIT_RATIO,0.118425
7,AGE_YEARS,0.089471
8,NAME_EDUCATION_TYPE,0.088716
9,EMPLOYED_AGE_RATIO,0.085571


,risk_group,rank_min,rank_max,n,bad_n,bad_rate,probability_min,probability_max,lift,cumulative_bad_capture
0,1,1,4613,4613,1289,0.279428,0.178880,0.812366,3.461107,0.346133
1,2,4614,9226,4613,679,0.147193,0.117450,0.178822,1.823190,0.528464
2,3,9227,13839,4613,483,0.104704,0.087189,0.117445,1.296908,0.658163
3,4,13840,18451,4612,384,0.083261,0.066852,0.087184,1.031306,0.761278
4,5,18452,23064,4613,292,0.063299,0.052413,0.066839,0.784052,0.839689
5,6,23065,27677,4613,196,0.042489,0.041657,0.052412,0.526282,0.892320
6,7,27678,32289,4612,154,0.033391,0.032553,0.041655,0.413597,0.933673
7,8,32290,36902,4613,117,0.025363,0.024800,0.032551,0.314158,0.965091
8,9,36903,41515,4613,77,0.016692,0.017018,0.024798,0.206753,0.985768
9,10,41516,46127,4612,53,0.011492,0.002356,0.017017,0.142342,1.000000


{'model_name': 'catboost',
 'experiment_track': 'controlled',
 'data_sha256': '52e96b895b1112e1c853f670e58372719c8441c5ed1c57ac2f7fad559d784f5f',
 'split_sha256': '4ae4ceee661ba8c210fa0a1cf950448799a0aef468659e6d2590c1bb4e8a3835',
 'feature_policy_version': 1,
 'training_sample': 'Train only',
 'selection_role': 'early stopping, candidate selection, and paired comparison only',
 'test_accessed': False,
 'parameters': {'iterations': 3770,
  'depth': 4,
  'learning_rate': 0.03,
  'l2_leaf_reg': 5.0,
  'subsample': 0.8,
  'rsm': 0.9,
  'random_strength': 0.5,
  'loss_function': 'Logloss',
  'eval_metric': 'AUC',
  'bootstrap_type': 'Bernoulli',
  'random_seed': 42,
  'thread_count': -1,
  'allow_writing_files': False,
  'verbose': False},
 'selected_profile': 'shallow_faster',
 'candidate_count': 9,
 'base_candidate_count': 6,
 'boundary_extensions': 3,
 'selected_candidate_reached_extension_boundary': False,
 'final_refit_on_train_only_at_fixed_iteration': True,
 'raw_feature_count': 126

锁模产物: /Users/ganlu/Desktop/home-credit-risk-english/artifacts/catboost/20260831T060333_256012Z


In [6]:
registry_path = PROJECT_ROOT / development.configs['experiment']['test_policy']['registry_path']
registry = json.loads(registry_path.read_text(encoding='utf-8'))
assert registry['models']['catboost']['locked'] is True
assert registry['models']['catboost']['config']['test_accessed'] is False
assert registry['suite_locked'] is False
print('CatBoost 已锁定；LightGBM、XGBoost、MLP 尚未全部登记，Test 继续封存。')

CatBoost 已锁定；LightGBM、XGBoost、MLP 尚未全部登记，Test 继续封存。
